## 01 — Exploratory single-fixture ingest from Sportmonks API
## 01 — Eksploracyjny ingest jednego meczu z API Sportmonks

**PL:** Notebook prototypowy. Pobiera jeden mecz z API Sportmonks v3, zapisuje surowy JSON do Unity Catalog Volume i weryfikuje jego strukturę w Spark.  
**EN:** Prototype notebook. Fetches a single fixture from the Sportmonks v3 API, writes the raw JSON to a Unity Catalog Volume, and validates its structure in Spark.

> ⚠️ **TOKEN** is read from Databricks Secret Scope `sportmonks` — never hardcode credentials.


In [ ]:
import requests
import json
from datetime import datetime, timezone

TOKEN = dbutils.secrets.get(
    scope="sportmonks",
    key="api-token"
)

fixture_id = 19499029
league_id = 45
season_id = 26097 

### 1. Imports & configuration / Importy i konfiguracja

**PL:** Importujemy biblioteki HTTP i JSON. `TOKEN` jest pobierany z Databricks Secret Scope, dzięki czemu poświadczenia nie są nigdzie zapisane w kodzie. `fixture_id`, `league_id` i `season_id` identyfikują konkretne spotkanie i sezon WSL.  
**EN:** Imports HTTP and JSON libraries. `TOKEN` is retrieved from Databricks Secret Scope so no credentials are embedded in the code. `fixture_id`, `league_id`, and `season_id` identify the specific WSL fixture and season.


In [ ]:
url = f"https://api.sportmonks.com/v3/football/fixtures/{fixture_id}"

includes = (
    "state;"
    "lineups;"
    "events;"
    "statistics;"
    "periods;"
    "participants;"
    "scores;"
    "formations;"
    "referees;"
    "sidelined;"
    "metadata;"
    "comments;"
    "timeline;"
    "tvStations;"
    "coaches;"
    "group;"
    "round;"
    "stage;"
    "season;"
    "venue;"
    "league;"
    "pressure;"
    "xGFixture"
)

params = {
    "api_token": TOKEN,
    "include": includes
}

### 2. Build API request / Budowanie zapytania do API

**PL:** Konstruujemy URL dla endpointu `/fixtures/{fixture_id}` i definiujemy listę `includes` — zagnieżdżonych relacji Sportmonks, które mają zostać zwrócone w jednym żądaniu (skład, zdarzenia, statystyki, indeks pressure, xG itp.). Jeden request z `include` jest tańszy (rate-limit) niż wiele osobnych zapytań.  
**EN:** Builds the URL for the `/fixtures/{fixture_id}` endpoint and defines the `includes` list — Sportmonks nested relations to embed in a single request (lineups, events, statistics, pressure index, xG, etc.). One request with `include` is cheaper (rate-limit) than multiple separate calls.


In [ ]:
response = requests.get(
    url,
    params = params,
    timeout = 30,
)

response.raise_for_status()

fixture_json = response.json()



### 3. Execute request / Wykonanie zapytania

**PL:** Wysyłamy GET request z timeoutem 30 s. `raise_for_status()` rzuca wyjątek dla kodów 4xx/5xx, co pozwala na wcześnie wykryć błędy autoryzacji lub rate-limiting.  
**EN:** Sends the GET request with a 30 s timeout. `raise_for_status()` raises on 4xx/5xx, catching auth errors and rate-limiting early.


In [ ]:
print("HTTP status:", response.status_code)
print("Fixture:", fixture_json["data"]["name"])
print("Fixture ID:", fixture_json["data"]["id"])

HTTP status: 200
Fixture: Arsenal W vs London City Lionesses W
Fixture ID: 19499029


### 4. Quick inspection / Szybka inspekcja odpowiedzi

**PL:** Wypisujemy kod HTTP i nazwę meczu jako szybka sanity-check przed zapisem.  
**EN:** Prints HTTP status and fixture name as a quick sanity-check before writing.


In [ ]:
fixture_json["data"].keys()

dict_keys(['id', 'sport_id', 'league_id', 'season_id', 'stage_id', 'group_id', 'aggregate_id', 'round_id', 'state_id', 'venue_id', 'name', 'starting_at', 'result_info', 'leg', 'details', 'length', 'placeholder', 'has_odds', 'has_premium_odds', 'starting_at_timestamp', 'state', 'lineups', 'events', 'statistics', 'periods', 'participants', 'scores', 'formations', 'referees', 'sidelined', 'metadata', 'comments', 'timeline', 'tvstations', 'coaches', 'group', 'round', 'stage', 'season', 'venue', 'league', 'pressure', 'xgfixture'])

### 5. List available keys / Lista kluczy w odpowiedzi

**PL:** Pokazujemy listę kluczy w `data`, żeby zobaczyć, które includes zostały faktycznie zwrócone przez API.  
**EN:** Lists keys present in `data` to verify which includes were actually returned by the API.


In [ ]:
ingestion_date = datetime.now(timezone.utc).date().isoformat()

output_dir = (
    "/Volumes/wsl_analytics/landing/sportmonks_raw/"
    f"fixtures/"
    f"league_id={league_id}/"
    f"season_id={season_id}/"
    f"ingestion_date={ingestion_date}"
)

output_dir

'/Volumes/wsl_analytics/landing/sportmonks_raw/fixtures/league_id=45/season_id=26097/ingestion_date=2026-08-13'

### 6. Build output path / Budowanie ścieżki wyjściowej

**PL:** Tworzymy ścieżkę w Unity Catalog Volume z partycjonowaniem Hive-style (`league_id=…/season_id=…/ingestion_date=…`). Data ingestii pochodzi z UTC, co zapewnia spójność między strefami czasowymi.  
**EN:** Builds a path in the Unity Catalog Volume using Hive-style partition directories (`league_id=…/season_id=…/ingestion_date=…`). The ingestion date is UTC to ensure timezone consistency.


In [ ]:
import os

os.makedirs(output_dir,exist_ok = True)

### 7. Create directory / Tworzenie katalogu

**PL:** `os.makedirs` z `exist_ok=True` — idempotentne tworzenie katalogu, bezpieczne przy ponownych uruchomieniach.  
**EN:** `os.makedirs` with `exist_ok=True` — idempotent directory creation, safe to re-run.


In [ ]:
output_path = f"{output_dir}/fixture_{fixture_id}.json"

with open(output_path, "w", encoding = "utf-8") as f:
    json.dump(
        fixture_json,
        f,
        ensure_ascii = False,
        indent = 2
    )

print(output_path)

/Volumes/wsl_analytics/landing/sportmonks_raw/fixtures/league_id=45/season_id=26097/ingestion_date=2026-08-13/fixture_19499029.json


### 8. Write JSON to Volume / Zapis JSON do Volume

**PL:** Zapisujemy pełną odpowiedź API jako sformatowany JSON. `ensure_ascii=False` zachowuje znaki Unicode (nazwy zawodniczek). `indent=2` ułatwia ręczną inspekcję pliku.  
**EN:** Writes the full API response as pretty-printed JSON. `ensure_ascii=False` preserves Unicode characters (player names). `indent=2` enables easy manual inspection.


In [ ]:
display(
    dbutils.fs.ls(output_dir)
)

path,name,size,modificationTime
dbfs:/Volumes/wsl_analytics/landing/sportmonks_raw/fixtures/league_id=45/season_id=26097/ingestion_date=2026-08-13/fixture_19499029.json,fixture_19499029.json,115165,1786581583000


### 9. Verify write / Weryfikacja zapisu

**PL:** `dbutils.fs.ls()` wypisuje zawartość katalogu — potwierdzamy, że plik istnieje i ma niezerowy rozmiar.  
**EN:** `dbutils.fs.ls()` lists the directory — confirms the file exists and has non-zero size.


In [ ]:
test_df = (
    spark.read
    .option("multiLine", "true")
    .json(output_path)
)

display(test_df)

data rate_limit subscription timezone List(null, List(List(null, R. Slegers, 38, 1988-02-05, Renee Slegers, Renee, female, null, 333232, https://cdn.sportmonks.com/images/soccer/players/16/333232.png, Slegers, List(333232, 19499029, 62847), Renee Slegers, 38, 333232, 1, null), List(null, G. Prêcheur, 17, 1959-10-23, Gérard Prêcheur, Gérard, male, null, 37654887, https://cdn.sportmonks.com/images/soccer/placeholder.png, Prêcheur, List(37654887, 19499029, 132782), Gérard Prêcheur, 17, 37654887, 1, null)), List(), null, List(List(1st Penalty, null, 6208321, null, 19499029, 151126218, Penalty, null, 17, false, 132782, 6208321, 752070, Kosovare Asllani, null, null, null, 0-1, event, 1, null, 16), List(1st Goal, null, 6208321, null, 19499029, 151126382, Right foot shot, null, 29, false, 62847, 6208321, 37400282, Olivia Smith, 752186, Stephanie Catley, null, 1-1, event, 2, 1522, 14), List(2nd Goal, null, 6208321, 2, 19499029, 151126692, Right foot shot, null, 45, false, 62847, 6208321, 25558238, Chloe Maggie Kelly, 37447078, Alessia Mia Teresa Russo, null, 2-1, event, 3, 1522, 14), List(3rd Goal, null, 6208570, null, 19499029, 151127933, Right foot shot, null, 83, false, 62847, 6208570, 333240, Emma Stina Blackstenius, 15061043, Bethany Jane Mead, null, 3-1, event, 4, 1522, 14), List(4th Goal, null, 6208570, null, 19499029, 151128027, Header, null, 84, false, 62847, 6208570, 4236997, Frida Leonhardsen Maanum, 15061043, Bethany Jane Mead, null, 4-1, event, 5, 1694, 14), List(1st Substitution, null, 6208570, null, 19499029, 151127026, null, null, 46, false, 132782, 6208570, 333364, Julia Elisabeth Roddar, 27431501, Katie Zelem, null, null, event, 1, null, 18), List(2nd Substitution, null, 6208570, null, 19499029, 151127376, null, null, 61, false, 62847, 6208570, 752176, Caitlin Foord, 37400282, Olivia Smith, null, null, event, 2, null, 18), List(3rd Substitution, null, 6208570, null, 19499029, 151127379, null, null, 61, false, 62847, 6208570, 4236997, Frida Leonhardsen Maanum, 10276134, Victoria Pelova, null, null, event, 3, null, 18), List(4th Substitution, null, 6208570, null, 19499029, 151127436, null, null, 65, false, 132782, 6208570, 8417711, Sanni Maija Franssi, 37574042, María Pérez, null, null, event, 4, null, 18), List(5th Substitution, null, 6208570, null, 19499029, 151127437, null, null, 65, false, 132782, 6208570, 752064, Eva Sofia Jakobsson, 7709149, Nikita Josephine Parris, null, null, event, 5, null, 18), List(6th Substitution, null, 6208570, null, 19499029, 151127698, null, null, 76, false, 62847, 6208570, 15061043, Bethany Jane Mead, 25558238, Chloe Maggie Kelly, null, null, event, 6, null, 18), List(7th Substitution, null, 6208570, null, 19499029, 151127699, null, null, 77, false, 62847, 6208570, 333240, Emma Stina Blackstenius, 37447078, Alessia Mia Teresa Russo, null, null, event, 7, null, 18), List(8th Substitution, null, 6208570, null, 19499029, 151128110, null, null, 83, false, 132782, 6208570, 37565037, Teyah Goldie, 752175, Alanna Kennedy, null, null, event, 8, null, 18), List(9th Substitution, null, 6208570, null, 19499029, 151128147, null, null, 87, false, 62847, 6208570, 27431562, Taylor Hinds, 8489308, Katie McCabe, null, null, event, 9, null, 18), List(10th Substitution, null, 6208570, 3, 19499029, 151128213, null, null, 90, false, 132782, 6208570, 37594960, Lucía Corrales Álvarez, 37604207, Rofiat Imuran, null, null, event, 10, null, 18), List(1st Yellowcard, null, 6208321, null, 19499029, 151126280, Foul, null, 20, false, 132782, 6208321, 7709149, Nikita Josephine Parris, null, null, false, null, event, 1, null, 19), List(2nd Yellowcard, null, 6208321, null, 19499029, 151126376, Foul, null, 28, false, 132782, 6208321, 27431501, Katie Zelem, null, null, false, null, event, 2, null, 19), List(3rd Yellowcard, null, 6208570, null, 19499029, 151127338, Foul, null, 58, false, 132782, 6208570, 8044227, Saki Kumagai, null, null, false, null, event, 3, null, 19), List(4th Yellowcard, null, 6208570, null, 19499029

### 10. Smoke test in Spark / Weryfikacja struktury w Spark

**PL:** Wczytujemy zapisany JSON do DataFrame z opcją `multiLine=True`, ponieważ plik jest wielowierszowym JSONem (nie JSONL). `display()` pokazuje zagnieżdżoną strukturę.  
**EN:** Reads the saved JSON into a DataFrame with `multiLine=True` since the file is pretty-printed JSON (not JSONL). `display()` reveals the nested structure.


In [ ]:
test_df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- aggregate_id: string (nullable = true)
 |    |-- coaches: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- city_id: string (nullable = true)
 |    |    |    |-- common_name: string (nullable = true)
 |    |    |    |-- country_id: long (nullable = true)
 |    |    |    |-- date_of_birth: string (nullable = true)
 |    |    |    |-- display_name: string (nullable = true)
 |    |    |    |-- firstname: string (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |    |-- height: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image_path: string (nullable = true)
 |    |    |    |-- lastname: string (nullable = true)
 |    |    |    |-- meta: struct (nullable = true)
 |    |    |    |    |-- coach_id: long (nullable = true)
 |    |    |    |    |-- fixture_id: long (nullable = true)
 |    |    |    |    |-- participant_i

### 11. Print schema / Schemat

**PL:** `printSchema()` pokazuje wykryte przez Spark typy danych — dobry punkt startowy do planowania warstwy bronze.  
**EN:** `printSchema()` shows Spark-inferred types — useful starting point for planning the bronze schema.


In [ ]:
from pyspark.sql import functions as F

test_df.select(
    F.col("data.id").alias("fixture_id"),
    F.col("data.name").alias("fixture_name"),
    F.size("data.events").alias("n_events"),
    F.size("data.statistics").alias("n_statistics"),
    F.size("data.participants").alias("n_participants"),
    F.size("data.pressure").alias("n_pressure")
).display()

fixture_id,fixture_name,n_events,n_statistics,n_participants,n_pressure
19499029,Arsenal W vs London City Lionesses W,19,78,2,192


### 12. Field-level counts / Liczba rekordów na pole

**PL:** Wybieramy kilka zagnieżdżonych pól i sprawdzamy, ile rekordów zawierają (`size()`). W szczególności `n_pressure` mówi nam, czy dla tego meczu dostępny jest indeks pressure — kluczowy dla dalszych analiz.  
**EN:** Selects a few nested fields and counts their array lengths (`size()`). `n_pressure` in particular tells us whether the pressure index is available for this fixture — critical for downstream analysis.


In [ ]:
test_df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- aggregate_id: string (nullable = true)
 |    |-- coaches: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- city_id: string (nullable = true)
 |    |    |    |-- common_name: string (nullable = true)
 |    |    |    |-- country_id: long (nullable = true)
 |    |    |    |-- date_of_birth: string (nullable = true)
 |    |    |    |-- display_name: string (nullable = true)
 |    |    |    |-- firstname: string (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |    |-- height: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image_path: string (nullable = true)
 |    |    |    |-- lastname: string (nullable = true)
 |    |    |    |-- meta: struct (nullable = true)
 |    |    |    |    |-- coach_id: long (nullable = true)
 |    |    |    |    |-- fixture_id: long (nullable = true)
 |    |    |    |    |-- participant_i